CELL 1 — Cài thư viện

In [1]:
# Cài các thư viện cần thiết cho Colab
!pip -q install duckdb pyarrow pandas fastparquet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 10.8 MB/s eta 0:00:00


CELL 2 — Khởi tạo thư mục, cấu hình dataset, tải dữ liệu gốc

In [2]:
import os
import urllib.request
from pathlib import Path

# =========================
# 1. CẤU HÌNH DATASET
# =========================
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"
DATASET_PREFIX = "yellow_tripdata"
YEAR = 2024
MONTHS = list(range(1, 13))  # tải đủ 12 tháng

# =========================
# 2. CẤU TRÚC THƯ MỤC VERSION
# =========================
PROJECT_DIR = Path("/content/project_bigdata_task1")
RAW_DIR = PROJECT_DIR / "data" / "raw"
V1_DIR = PROJECT_DIR / "data" / "v1_loaded"
V2_DIR = PROJECT_DIR / "data" / "v2_cleaned_nulls"
V3_DIR = PROJECT_DIR / "data" / "v3_deduplicated"
V4_DIR = PROJECT_DIR / "data" / "v4_validated"
V5_DIR = PROJECT_DIR / "data" / "v5_normalized"
FINAL_DIR = PROJECT_DIR / "data" / "final"
REPORT_DIR = PROJECT_DIR / "reports"

for folder in [RAW_DIR, V1_DIR, V2_DIR, V3_DIR, V4_DIR, V5_DIR, FINAL_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)

# =========================
# 3. TẢI DỮ LIỆU GỐC
#    Lưu ý: chỉ tải vào RAW_DIR
#    và KHÔNG BAO GIỜ ghi đè xử lý lên raw
# =========================
downloaded_files = []

for month in MONTHS:
    mm = f"{month:02d}"
    filename = f"{DATASET_PREFIX}_{YEAR}-{mm}.parquet"
    url = f"{BASE_URL}/{filename}"
    save_path = RAW_DIR / filename

    if not save_path.exists():
        print(f"Downloading: {filename}")
        urllib.request.urlretrieve(url, save_path)
    else:
        print(f"Exists: {filename}")

    downloaded_files.append(str(save_path))

print("\nDownloaded files:", len(downloaded_files))
print("Sample raw file:", downloaded_files[0])

Project directory: /content/project_bigdata_task1
Downloading: yellow_tripdata_2024-01.parquet
Downloading: yellow_tripdata_2024-02.parquet
Downloading: yellow_tripdata_2024-03.parquet
Downloading: yellow_tripdata_2024-04.parquet
Downloading: yellow_tripdata_2024-05.parquet
Downloading: yellow_tripdata_2024-06.parquet
Downloading: yellow_tripdata_2024-07.parquet
Downloading: yellow_tripdata_2024-08.parquet
Downloading: yellow_tripdata_2024-09.parquet
Downloading: yellow_tripdata_2024-10.parquet
Downloading: yellow_tripdata_2024-11.parquet
Downloading: yellow_tripdata_2024-12.parquet

Downloaded files: 12
Sample raw file: /content/project_bigdata_task1/data/raw/yellow_tripdata_2024-01.parquet


CELL 3 — Đọc schema, xem mẫu dữ liệu, lưu VERSION 1

In [3]:
import duckdb
import pandas as pd
import os

# Kết nối DuckDB
con = duckdb.connect()

raw_glob = str(RAW_DIR / "*.parquet")
v1_file = str(V1_DIR / "v1_loaded.parquet")

# =========================
# 1. XEM SCHEMA TỪ DỮ LIỆU GỐC
# =========================
schema_df = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{raw_glob}')
""").fetchdf()

print("=== SCHEMA OF RAW DATA ===")
display(schema_df)

# =========================
# 2. XEM 10 DÒNG MẪU
# =========================
sample_df = con.execute(f"""
SELECT * FROM read_parquet('{raw_glob}')
LIMIT 10
""").fetchdf()

print("=== SAMPLE RAW DATA ===")
display(sample_df)

# =========================
# 3. ĐẾM TỔNG SỐ DÒNG DỮ LIỆU GỐC
# =========================
raw_count_df = con.execute(f"""
SELECT COUNT(*) AS raw_total_rows
FROM read_parquet('{raw_glob}')
""").fetchdf()

print("=== RAW TOTAL ROWS ===")
display(raw_count_df)

# =========================
# 4. LƯU VERSION 1
#    Mục tiêu: lưu lại trạng thái sau khi load dữ liệu
#    hoàn toàn không chỉnh sửa nội dung dữ liệu gốc
# =========================
con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{raw_glob}')
) TO '{v1_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved V1:", v1_file)

=== SCHEMA OF RAW DATA ===


,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


=== SAMPLE RAW DATA ===


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee
0,2,2024-01-01 00:57:55,2024-01-01 01:17:43,1,1.72,1,N,186,79,2,17.7,1.0,0.5,0.00,0.0,1.0,22.70,2.5,0.00
1,1,2024-01-01 00:03:00,2024-01-01 00:09:36,1,1.80,1,N,140,236,1,10.0,3.5,0.5,3.75,0.0,1.0,18.75,2.5,0.00
2,1,2024-01-01 00:17:06,2024-01-01 00:35:01,1,4.70,1,N,236,79,1,23.3,3.5,0.5,3.00,0.0,1.0,31.30,2.5,0.00
3,1,2024-01-01 00:36:38,2024-01-01 00:44:56,1,1.40,1,N,79,211,1,10.0,3.5,0.5,2.00,0.0,1.0,17.00,2.5,0.00
4,1,2024-01-01 00:46:51,2024-01-01 00:52:57,1,0.80,1,N,211,148,1,7.9,3.5,0.5,3.20,0.0,1.0,16.10,2.5,0.00
5,1,2024-01-01 00:54:08,2024-01-01 01:26:31,1,4.70,1,N,148,141,1,29.6,3.5,0.5,6.90,0.0,1.0,41.50,2.5,0.00
6,2,2024-01-01 00:49:44,2024-01-01 01:15:47,2,10.82,1,N,138,181,1,45.7,6.0,0.5,10.00,0.0,1.0,64.95,0.0,1.75
7,1,2024-01-01 00:30:40,2024-01-01 00:58:40,0,3.00,1,N,246,231,2,25.4,3.5,0.5,0.00,0.0,1.0,30.40,2.5,0.00
8,2,2024-01-01 00:26:01,2024-01-01 00:54:12,1,5.44,1,N,161,261,2,31.0,1.0,0.5,0.00,0.0,1.0,36.00,2.5,0.00
9,2,2024-01-01 00:28:08,2024-01-01 00:29:16,1,0.04,1,N,113,113,2,3.0,1.0,0.5,0.00,0.0,1.0,8.00,2.5,0.00


=== RAW TOTAL ROWS ===


,raw_total_rows
0,41169720


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V1: /content/project_bigdata_task1/data/v1_loaded/v1_loaded.parquet


CELL 4 — Kiểm tra NULL/NaN, xử lý dữ liệu thiếu, lưu VERSION 2

In [4]:
# File input/output cho bước này
v1_file = str(V1_DIR / "v1_loaded.parquet")
v2_file = str(V2_DIR / "v2_cleaned_nulls.parquet")

# =========================
# 1. KIỂM TRA NULL TRÊN CÁC CỘT CHÍNH
# =========================
null_report_df = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS VendorID_nulls,
    SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS pickup_datetime_nulls,
    SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS dropoff_datetime_nulls,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_nulls,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance_nulls,
    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS PULocationID_nulls,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS DOLocationID_nulls,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount_nulls,
    SUM(CASE WHEN store_and_fwd_flag IS NULL THEN 1 ELSE 0 END) AS store_and_fwd_flag_nulls,
    SUM(CASE WHEN RatecodeID IS NULL THEN 1 ELSE 0 END) AS RatecodeID_nulls
FROM read_parquet('{v1_file}')
""").fetchdf()

print("=== NULL REPORT BEFORE CLEANING ===")
display(null_report_df)

# Lưu report ra CSV để có bằng chứng từng bước
null_report_path = REPORT_DIR / "null_report_before_cleaning.csv"
null_report_df.to_csv(null_report_path, index=False)
print("Saved report:", null_report_path)

# =========================
# 2. XỬ LÝ DỮ LIỆU THIẾU
#
# QUY TẮC:
# - XÓA dòng nếu thiếu cột lõi bắt buộc:
#   VendorID, pickup datetime, dropoff datetime,
#   PULocationID, DOLocationID, payment_type, total_amount
#
# - FILL hợp lý nếu thiếu cột phụ:
#   passenger_count -> 1
#   trip_distance -> 0
#   RatecodeID -> 99
#   store_and_fwd_flag -> 'N'
#   các khoản phụ phí -> 0
# =========================
con.execute(f"""
COPY (
    SELECT
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,

        COALESCE(passenger_count, 1) AS passenger_count,
        COALESCE(trip_distance, 0) AS trip_distance,
        COALESCE(RatecodeID, 99) AS RatecodeID,
        COALESCE(store_and_fwd_flag, 'N') AS store_and_fwd_flag,

        PULocationID,
        DOLocationID,
        payment_type,

        COALESCE(fare_amount, 0) AS fare_amount,
        COALESCE(extra, 0) AS extra,
        COALESCE(mta_tax, 0) AS mta_tax,
        COALESCE(tip_amount, 0) AS tip_amount,
        COALESCE(tolls_amount, 0) AS tolls_amount,
        COALESCE(improvement_surcharge, 0) AS improvement_surcharge,
        COALESCE(total_amount, 0) AS total_amount,
        COALESCE(congestion_surcharge, 0) AS congestion_surcharge,
        COALESCE(Airport_fee, 0) AS Airport_fee
    FROM read_parquet('{v1_file}')
    WHERE
        VendorID IS NOT NULL
        AND tpep_pickup_datetime IS NOT NULL
        AND tpep_dropoff_datetime IS NOT NULL
        AND PULocationID IS NOT NULL
        AND DOLocationID IS NOT NULL
        AND payment_type IS NOT NULL
        AND total_amount IS NOT NULL
) TO '{v2_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved V2:", v2_file)

# =========================
# 3. KIỂM TRA LẠI NULL SAU XỬ LÝ
# =========================
null_after_df = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN VendorID IS NULL THEN 1 ELSE 0 END) AS VendorID_nulls,
    SUM(CASE WHEN tpep_pickup_datetime IS NULL THEN 1 ELSE 0 END) AS pickup_datetime_nulls,
    SUM(CASE WHEN tpep_dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS dropoff_datetime_nulls,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_nulls,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance_nulls,
    SUM(CASE WHEN PULocationID IS NULL THEN 1 ELSE 0 END) AS PULocationID_nulls,
    SUM(CASE WHEN DOLocationID IS NULL THEN 1 ELSE 0 END) AS DOLocationID_nulls,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount_nulls
FROM read_parquet('{v2_file}')
""").fetchdf()

print("=== NULL REPORT AFTER CLEANING ===")
display(null_after_df)

null_after_path = REPORT_DIR / "null_report_after_cleaning.csv"
null_after_df.to_csv(null_after_path, index=False)
print("Saved report:", null_after_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== NULL REPORT BEFORE CLEANING ===


,total_rows,VendorID_nulls,pickup_datetime_nulls,dropoff_datetime_nulls,passenger_count_nulls,trip_distance_nulls,PULocationID_nulls,DOLocationID_nulls,payment_type_nulls,total_amount_nulls,store_and_fwd_flag_nulls,RatecodeID_nulls
0,41169720,0.0,0.0,0.0,4091232.0,0.0,0.0,0.0,0.0,0.0,4091232.0,4091232.0


Saved report: /content/project_bigdata_task1/reports/null_report_before_cleaning.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V2: /content/project_bigdata_task1/data/v2_cleaned_nulls/v2_cleaned_nulls.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== NULL REPORT AFTER CLEANING ===


,total_rows,VendorID_nulls,pickup_datetime_nulls,dropoff_datetime_nulls,passenger_count_nulls,trip_distance_nulls,PULocationID_nulls,DOLocationID_nulls,payment_type_nulls,total_amount_nulls
0,41169720,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Saved report: /content/project_bigdata_task1/reports/null_report_after_cleaning.csv


CELL 5 — Xóa duplicate, lưu VERSION 3

In [5]:
v2_file = str(V2_DIR / "v2_cleaned_nulls.parquet")
v3_file = str(V3_DIR / "v3_deduplicated.parquet")

# =========================
# 1. KIỂM TRA DUPLICATE TRƯỚC KHI XÓA
# =========================
dup_report_before = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        PULocationID,
        DOLocationID,
        payment_type,
        total_amount
    )) AS duplicate_rows
FROM read_parquet('{v2_file}')
""").fetchdf()

print("=== DUPLICATE REPORT BEFORE ===")
display(dup_report_before)

dup_before_path = REPORT_DIR / "duplicate_report_before.csv"
dup_report_before.to_csv(dup_before_path, index=False)
print("Saved report:", dup_before_path)

# =========================
# 2. XÓA DUPLICATE
#    Dùng DISTINCT trên toàn bộ record đã qua bước xử lý null
# =========================
con.execute(f"""
COPY (
    SELECT DISTINCT *
    FROM read_parquet('{v2_file}')
) TO '{v3_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved V3:", v3_file)

# =========================
# 3. KIỂM TRA LẠI DUPLICATE SAU KHI XÓA
# =========================
dup_report_after = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        VendorID,
        tpep_pickup_datetime,
        tpep_dropoff_datetime,
        passenger_count,
        trip_distance,
        PULocationID,
        DOLocationID,
        payment_type,
        total_amount
    )) AS duplicate_rows
FROM read_parquet('{v3_file}')
""").fetchdf()

print("=== DUPLICATE REPORT AFTER ===")
display(dup_report_after)

dup_after_path = REPORT_DIR / "duplicate_report_after.csv"
dup_report_after.to_csv(dup_after_path, index=False)
print("Saved report:", dup_after_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DUPLICATE REPORT BEFORE ===


,total_rows,duplicate_rows
0,41169720,5


Saved report: /content/project_bigdata_task1/reports/duplicate_report_before.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V3: /content/project_bigdata_task1/data/v3_deduplicated/v3_deduplicated.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== DUPLICATE REPORT AFTER ===


,total_rows,duplicate_rows
0,41169716,1


Saved report: /content/project_bigdata_task1/reports/duplicate_report_after.csv
Error: Runtime no longer has a reference to this dataframe, please re-run this cell and try again.


CELL 6 — Loại dữ liệu sai, giá trị âm vô lý, lỗi thời gian, lưu VERSION 4

In [6]:
v3_file = str(V3_DIR / "v3_deduplicated.parquet")
v4_file = str(V4_DIR / "v4_validated.parquet")

# =========================
# 1. BÁO CÁO DỮ LIỆU SAI TRƯỚC KHI XỬ LÝ
# =========================
invalid_before_df = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN CAST(tpep_dropoff_datetime AS TIMESTAMP) < CAST(tpep_pickup_datetime AS TIMESTAMP) THEN 1 ELSE 0 END) AS bad_time_rows,
    SUM(CASE WHEN COALESCE(passenger_count, 0) < 0 THEN 1 ELSE 0 END) AS negative_passenger_rows,
    SUM(CASE WHEN COALESCE(trip_distance, 0) < 0 THEN 1 ELSE 0 END) AS negative_trip_distance_rows,
    SUM(CASE WHEN COALESCE(fare_amount, 0) < 0 THEN 1 ELSE 0 END) AS negative_fare_rows,
    SUM(CASE WHEN COALESCE(extra, 0) < 0 THEN 1 ELSE 0 END) AS negative_extra_rows,
    SUM(CASE WHEN COALESCE(mta_tax, 0) < 0 THEN 1 ELSE 0 END) AS negative_mta_tax_rows,
    SUM(CASE WHEN COALESCE(tip_amount, 0) < 0 THEN 1 ELSE 0 END) AS negative_tip_rows,
    SUM(CASE WHEN COALESCE(tolls_amount, 0) < 0 THEN 1 ELSE 0 END) AS negative_tolls_rows,
    SUM(CASE WHEN COALESCE(improvement_surcharge, 0) < 0 THEN 1 ELSE 0 END) AS negative_improvement_rows,
    SUM(CASE WHEN COALESCE(total_amount, 0) < 0 THEN 1 ELSE 0 END) AS negative_total_rows,
    SUM(CASE WHEN COALESCE(congestion_surcharge, 0) < 0 THEN 1 ELSE 0 END) AS negative_congestion_rows,
    SUM(CASE WHEN COALESCE(Airport_fee, 0) < 0 THEN 1 ELSE 0 END) AS negative_airport_fee_rows
FROM read_parquet('{v3_file}')
""").fetchdf()

print("=== INVALID DATA REPORT BEFORE ===")
display(invalid_before_df)

invalid_before_path = REPORT_DIR / "invalid_report_before.csv"
invalid_before_df.to_csv(invalid_before_path, index=False)
print("Saved report:", invalid_before_path)

# =========================
# 2. LOẠI BỎ DỮ LIỆU SAI
#
# QUY TẮC:
# - dropoff < pickup => loại
# - số âm vô lý => loại
# - ràng buộc hợp lý để bớt dữ liệu lỗi nặng/outlier bất thường
# =========================
con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{v3_file}')
    WHERE
        CAST(tpep_dropoff_datetime AS TIMESTAMP) >= CAST(tpep_pickup_datetime AS TIMESTAMP)
        AND passenger_count >= 0
        AND trip_distance >= 0
        AND fare_amount >= 0
        AND extra >= 0
        AND mta_tax >= 0
        AND tip_amount >= 0
        AND tolls_amount >= 0
        AND improvement_surcharge >= 0
        AND total_amount >= 0
        AND congestion_surcharge >= 0
        AND Airport_fee >= 0

        -- Một số ràng buộc thực tế để giảm bản ghi dị thường
        AND passenger_count <= 8
        AND trip_distance <= 200
        AND total_amount <= 1000
) TO '{v4_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved V4:", v4_file)

# =========================
# 3. BÁO CÁO SAU KHI VALIDATE
# =========================
invalid_after_df = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN CAST(tpep_dropoff_datetime AS TIMESTAMP) < CAST(tpep_pickup_datetime AS TIMESTAMP) THEN 1 ELSE 0 END) AS bad_time_rows,
    SUM(CASE WHEN passenger_count < 0 THEN 1 ELSE 0 END) AS negative_passenger_rows,
    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS negative_trip_distance_rows,
    SUM(CASE WHEN fare_amount < 0 THEN 1 ELSE 0 END) AS negative_fare_rows,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS negative_total_rows
FROM read_parquet('{v4_file}')
""").fetchdf()

print("=== INVALID DATA REPORT AFTER ===")
display(invalid_after_df)

invalid_after_path = REPORT_DIR / "invalid_report_after.csv"
invalid_after_df.to_csv(invalid_after_path, index=False)
print("Saved report:", invalid_after_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== INVALID DATA REPORT BEFORE ===


,total_rows,bad_time_rows,negative_passenger_rows,negative_trip_distance_rows,negative_fare_rows,negative_extra_rows,negative_mta_tax_rows,negative_tip_rows,negative_tolls_rows,negative_improvement_rows,negative_total_rows,negative_congestion_rows,negative_airport_fee_rows
0,41169716,1575.0,0.0,0.0,731024.0,304569.0,583164.0,1331.0,49272.0,606112.0,609344.0,495745.0,99253.0


Saved report: /content/project_bigdata_task1/reports/invalid_report_before.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V4: /content/project_bigdata_task1/data/v4_validated/v4_validated.parquet


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== INVALID DATA REPORT AFTER ===


,total_rows,bad_time_rows,negative_passenger_rows,negative_trip_distance_rows,negative_fare_rows,negative_total_rows
0,40432321,0.0,0.0,0.0,0.0,0.0


Saved report: /content/project_bigdata_task1/reports/invalid_report_after.csv


CELL 7 — Chuẩn hóa kiểu dữ liệu, chuẩn hóa text, thêm cột dẫn xuất, lưu VERSION 5

In [7]:
v4_file = str(V4_DIR / "v4_validated.parquet")
v5_file = str(V5_DIR / "v5_normalized.parquet")

# =========================
# 1. CHUẨN HÓA KIỂU DỮ LIỆU
#    - string -> number
#    - date -> datetime
#    - chuẩn hóa text flag
#    - làm tròn số
#    - thêm cột thời gian phục vụ phân tích sau
# =========================
con.execute(f"""
COPY (
    SELECT
        CAST(VendorID AS BIGINT) AS vendor_id,
        CAST(tpep_pickup_datetime AS TIMESTAMP) AS pickup_datetime,
        CAST(tpep_dropoff_datetime AS TIMESTAMP) AS dropoff_datetime,

        CAST(passenger_count AS BIGINT) AS passenger_count,
        CAST(trip_distance AS DOUBLE) AS trip_distance,
        CAST(RatecodeID AS BIGINT) AS rate_code_id,

        CASE
            WHEN UPPER(TRIM(CAST(store_and_fwd_flag AS VARCHAR))) IN ('Y', 'N')
                THEN UPPER(TRIM(CAST(store_and_fwd_flag AS VARCHAR)))
            ELSE 'N'
        END AS store_and_fwd_flag,

        CAST(PULocationID AS BIGINT) AS pu_location_id,
        CAST(DOLocationID AS BIGINT) AS do_location_id,
        CAST(payment_type AS BIGINT) AS payment_type,

        ROUND(CAST(fare_amount AS DOUBLE), 2) AS fare_amount,
        ROUND(CAST(extra AS DOUBLE), 2) AS extra,
        ROUND(CAST(mta_tax AS DOUBLE), 2) AS mta_tax,
        ROUND(CAST(tip_amount AS DOUBLE), 2) AS tip_amount,
        ROUND(CAST(tolls_amount AS DOUBLE), 2) AS tolls_amount,
        ROUND(CAST(improvement_surcharge AS DOUBLE), 2) AS improvement_surcharge,
        ROUND(CAST(total_amount AS DOUBLE), 2) AS total_amount,
        ROUND(CAST(congestion_surcharge AS DOUBLE), 2) AS congestion_surcharge,
        ROUND(CAST(Airport_fee AS DOUBLE), 2) AS airport_fee,

        DATE(CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_date,
        EXTRACT(YEAR FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_year,
        EXTRACT(MONTH FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_month,
        EXTRACT(DAY FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_day,
        EXTRACT(HOUR FROM CAST(tpep_pickup_datetime AS TIMESTAMP)) AS pickup_hour,

        DATE(CAST(tpep_dropoff_datetime AS TIMESTAMP)) AS dropoff_date
    FROM read_parquet('{v4_file}')
) TO '{v5_file}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print("Saved V5:", v5_file)

# =========================
# 2. XEM SCHEMA SAU CHUẨN HÓA
# =========================
normalized_schema_df = con.execute(f"""
DESCRIBE SELECT * FROM read_parquet('{v5_file}')
""").fetchdf()

print("=== NORMALIZED SCHEMA ===")
display(normalized_schema_df)

normalized_schema_path = REPORT_DIR / "normalized_schema.csv"
normalized_schema_df.to_csv(normalized_schema_path, index=False)
print("Saved report:", normalized_schema_path)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Saved V5: /content/project_bigdata_task1/data/v5_normalized/v5_normalized.parquet
=== NORMALIZED SCHEMA ===


,column_name,column_type,null,key,default,extra
0,vendor_id,BIGINT,YES,None,None,None
1,pickup_datetime,TIMESTAMP,YES,None,None,None
2,dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,rate_code_id,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,pu_location_id,BIGINT,YES,None,None,None
8,do_location_id,BIGINT,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


Saved report: /content/project_bigdata_task1/reports/normalized_schema.csv


CELL 8 — Kiểm tra cuối cùng: null, duplicate, consistency

In [8]:
v5_file = str(V5_DIR / "v5_normalized.parquet")

# =========================
# 1. KIỂM TRA NULL CUỐI
# =========================
final_null_check_df = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN vendor_id IS NULL THEN 1 ELSE 0 END) AS vendor_id_nulls,
    SUM(CASE WHEN pickup_datetime IS NULL THEN 1 ELSE 0 END) AS pickup_datetime_nulls,
    SUM(CASE WHEN dropoff_datetime IS NULL THEN 1 ELSE 0 END) AS dropoff_datetime_nulls,
    SUM(CASE WHEN passenger_count IS NULL THEN 1 ELSE 0 END) AS passenger_count_nulls,
    SUM(CASE WHEN trip_distance IS NULL THEN 1 ELSE 0 END) AS trip_distance_nulls,
    SUM(CASE WHEN pu_location_id IS NULL THEN 1 ELSE 0 END) AS pu_location_id_nulls,
    SUM(CASE WHEN do_location_id IS NULL THEN 1 ELSE 0 END) AS do_location_id_nulls,
    SUM(CASE WHEN payment_type IS NULL THEN 1 ELSE 0 END) AS payment_type_nulls,
    SUM(CASE WHEN total_amount IS NULL THEN 1 ELSE 0 END) AS total_amount_nulls
FROM read_parquet('{v5_file}')
""").fetchdf()

print("=== FINAL NULL CHECK ===")
display(final_null_check_df)

# =========================
# 2. KIỂM TRA DUPLICATE CUỐI
# =========================
final_dup_check_df = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) - COUNT(DISTINCT (
        vendor_id,
        pickup_datetime,
        dropoff_datetime,
        passenger_count,
        trip_distance,
        pu_location_id,
        do_location_id,
        payment_type,
        total_amount
    )) AS duplicate_rows
FROM read_parquet('{v5_file}')
""").fetchdf()

print("=== FINAL DUPLICATE CHECK ===")
display(final_dup_check_df)

# =========================
# 3. KIỂM TRA CONSISTENCY CUỐI
# =========================
final_consistency_df = con.execute(f"""
SELECT
    COUNT(*) AS total_rows,
    SUM(CASE WHEN dropoff_datetime < pickup_datetime THEN 1 ELSE 0 END) AS bad_time_rows,
    SUM(CASE WHEN passenger_count < 0 THEN 1 ELSE 0 END) AS bad_passenger_rows,
    SUM(CASE WHEN trip_distance < 0 THEN 1 ELSE 0 END) AS bad_trip_distance_rows,
    SUM(CASE WHEN total_amount < 0 THEN 1 ELSE 0 END) AS bad_total_rows
FROM read_parquet('{v5_file}')
""").fetchdf()

print("=== FINAL CONSISTENCY CHECK ===")
display(final_consistency_df)

# =========================
# 4. LƯU CÁC BÁO CÁO CUỐI
# =========================
final_null_check_df.to_csv(REPORT_DIR / "final_null_check.csv", index=False)
final_dup_check_df.to_csv(REPORT_DIR / "final_duplicate_check.csv", index=False)
final_consistency_df.to_csv(REPORT_DIR / "final_consistency_check.csv", index=False)

print("Saved final validation reports to:", REPORT_DIR)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== FINAL NULL CHECK ===


,total_rows,vendor_id_nulls,pickup_datetime_nulls,dropoff_datetime_nulls,passenger_count_nulls,trip_distance_nulls,pu_location_id_nulls,do_location_id_nulls,payment_type_nulls,total_amount_nulls
0,40432321,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== FINAL DUPLICATE CHECK ===


,total_rows,duplicate_rows
0,40432321,1


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== FINAL CONSISTENCY CHECK ===


,total_rows,bad_time_rows,bad_passenger_rows,bad_trip_distance_rows,bad_total_rows
0,40432321,0.0,0.0,0.0,0.0


Saved final validation reports to: /content/project_bigdata_task1/reports


CELL 9 — Xuất output cuối cùng ra Parquet và CSV

In [9]:
v5_file = str(V5_DIR / "v5_normalized.parquet")
final_parquet = str(FINAL_DIR / "final_cleaned_yellow_taxi_2024.parquet")
final_csv = str(FINAL_DIR / "final_cleaned_yellow_taxi_2024.csv")

# =========================
# 1. XUẤT FILE CUỐI DẠNG PARQUET
# =========================
con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{v5_file}')
) TO '{final_parquet}' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# =========================
# 2. XUẤT FILE CUỐI DẠNG CSV
# =========================
con.execute(f"""
COPY (
    SELECT *
    FROM read_parquet('{v5_file}')
) TO '{final_csv}' (HEADER, DELIMITER ',')
""")

print("Final Parquet:", final_parquet)
print("Final CSV:", final_csv)

# =========================
# 3. THỐNG KÊ SỐ DÒNG CUỐI
# =========================
final_count_df = con.execute(f"""
SELECT COUNT(*) AS final_rows
FROM read_parquet('{final_parquet}')
""").fetchdf()

print("=== FINAL ROW COUNT ===")
display(final_count_df)

# =========================
# 4. XEM MẪU DỮ LIỆU CUỐI
# =========================
final_sample_df = con.execute(f"""
SELECT *
FROM read_parquet('{final_parquet}')
LIMIT 10
""").fetchdf()

print("=== FINAL SAMPLE DATA ===")
display(final_sample_df)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Final Parquet: /content/project_bigdata_task1/data/final/final_cleaned_yellow_taxi_2024.parquet
Final CSV: /content/project_bigdata_task1/data/final/final_cleaned_yellow_taxi_2024.csv
=== FINAL ROW COUNT ===


,final_rows
0,40432321


=== FINAL SAMPLE DATA ===


,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,trip_distance,rate_code_id,store_and_fwd_flag,pu_location_id,do_location_id,payment_type,...,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,pickup_date,pickup_year,pickup_month,pickup_day,pickup_hour,dropoff_date
0,1,2024-01-02 18:06:11,2024-01-02 18:19:39,1,2.50,1,N,162,263,2,...,1.0,20.70,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
1,2,2024-01-02 18:30:54,2024-01-02 18:42:29,3,1.49,1,N,230,234,2,...,1.0,17.90,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
2,1,2024-01-02 18:46:38,2024-01-02 19:09:33,1,9.20,1,N,138,68,1,...,1.0,68.99,2.5,1.75,2024-01-02,2024,1,2,18,2024-01-02
3,1,2024-01-02 18:26:31,2024-01-02 18:40:40,1,1.50,1,N,68,164,2,...,1.0,19.30,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
4,2,2024-01-02 18:01:06,2024-01-02 18:18:39,1,3.38,1,N,186,13,1,...,1.0,30.72,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
5,2,2024-01-02 18:27:58,2024-01-02 18:37:11,1,3.70,1,N,88,79,1,...,1.0,28.20,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
6,2,2024-01-02 18:40:46,2024-01-02 18:53:00,1,2.38,1,N,162,43,1,...,1.0,25.68,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
7,2,2024-01-02 18:53:06,2024-01-02 19:36:38,3,14.85,1,N,68,10,2,...,1.0,74.54,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
8,2,2024-01-02 18:51:13,2024-01-02 19:08:23,1,4.82,1,N,114,142,1,...,1.0,37.25,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02
9,1,2024-01-02 18:39:32,2024-01-02 18:43:26,6,0.40,1,N,236,263,1,...,1.0,15.10,2.5,0.00,2024-01-02,2024,1,2,18,2024-01-02


CELL 10 — Tải file kết quả về máy

In [10]:
from google.colab import files

# Tải file parquet cuối
files.download('/content/project_bigdata_task1/data/final/final_cleaned_yellow_taxi_2024.parquet')

# Tải file csv cuối
files.download('/content/project_bigdata_task1/data/final/final_cleaned_yellow_taxi_2024.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>